# Phase 4 (CJPE) — c-1 decoder-only: BitFit
Run the cells top to bottom. Set every `None` in Cell E (hyperparameters) before running it. EPOCHS = 1 and training is capped at MAX_TRAIN_MINUTES = 60.

## Cell 0a — imports + HF login

In [11]:
# --- Phase 1 c-1 notebook, cell 1 (verbatim) ---
from collections import Counter
from tqdm.auto import tqdm
import pickle
import re
from collections import Counter, defaultdict


# HF login (Phase 1 cell 2), needed if the dataset download asks for auth
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## Cell 0b — tokenizer (Phase 1)

In [12]:
# --- Phase 1 c-1 notebook, cell 8 (verbatim) ---


SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]  # fixed ids 0-3, never merged

class BPETokenizer:
    def __init__(self):
        self.id_to_bytes = {}
        self.bytes_to_id = {}
        self.merges = {}
        self.merge_order = []
        self.merge_ranks = {}  # Precomputed pair -> rank for fast encoding
        self.cache = {}        # Chunk lookup cache
        self._vocab_size = 0
        
        # Splits on whitespace+word or just whitespace. 
        # Guarantees round-trip: "".join(chunks) == text
        self.split_pattern = re.compile(r'\s*\S+|\s+')

    def train(self, text: str, vocab_size: int):
        # 1. Setup base vocabulary
        for i, tok in enumerate(SPECIAL_TOKENS):
            b = tok.encode("utf-8")
            self.id_to_bytes[i] = b
            self.bytes_to_id[b] = i

        next_id = len(SPECIAL_TOKENS)
        byte_to_base_id = {}
        for value in range(256):
            b = bytes([value])
            self.id_to_bytes[next_id] = b
            self.bytes_to_id[b] = next_id
            byte_to_base_id[value] = next_id
            next_id += 1

        # 2. Split text into chunks and get frequencies
        chunks = self.split_pattern.findall(text)
        chunk_counts = Counter(chunks)
        
        # 3. Store each distinct chunk as a tuple of base byte ids
        chunk_to_ids = {
            chunk: tuple(byte_to_base_id[b] for b in chunk.encode("utf-8"))
            for chunk in chunk_counts
        }

        # 4. Initialize weighted pair counts and the pair-to-chunk index
        pair_counts = Counter()
        pair_to_chunks = defaultdict(set)
        
        def get_pairs(ids):
            return [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]

        for chunk, ids in chunk_to_ids.items():
            count = chunk_counts[chunk]
            for pair in get_pairs(ids):
                pair_counts[pair] += count
                pair_to_chunks[pair].add(chunk)

        target = vocab_size
        num_merges = target - next_id
        pbar = tqdm(total=num_merges, desc="Training BPE")

        # 5. Iteratively merge the best pair
        while next_id < target:
            if not pair_counts:
                break

            max_freq = max(pair_counts.values())
            if max_freq <= 1:
                break

            # Min tuple tie-breaker matching your original behavior
            best_pair = min(p for p, c in pair_counts.items() if c == max_freq)

            new_id = next_id
            self.merges[best_pair] = new_id
            self.merge_order.append(best_pair)
            self.merge_ranks[best_pair] = len(self.merge_order) - 1
            
            self.id_to_bytes[new_id] = self.id_to_bytes[best_pair[0]] + self.id_to_bytes[best_pair[1]]
            self.bytes_to_id[self.id_to_bytes[new_id]] = new_id

            # 6. Update only the chunks containing the merged pair
            chunks_to_update = list(pair_to_chunks[best_pair])
            for chunk in chunks_to_update:
                old_ids = chunk_to_ids[chunk]
                count = chunk_counts[chunk]
                
                # Subtract old pair combinations
                for pair in get_pairs(old_ids):
                    pair_counts[pair] -= count
                    if pair_counts[pair] <= 0:
                        del pair_counts[pair]
                    if chunk in pair_to_chunks[pair]:
                        pair_to_chunks[pair].remove(chunk)
                
                # Apply the merge inside this chunk
                new_ids = []
                i = 0
                while i < len(old_ids):
                    if i < len(old_ids) - 1 and (old_ids[i], old_ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(old_ids[i])
                        i += 1
                new_ids = tuple(new_ids)
                chunk_to_ids[chunk] = new_ids
                
                # Add new pair combinations created by the merge
                for pair in get_pairs(new_ids):
                    pair_counts[pair] += count
                    pair_to_chunks[pair].add(chunk)

            next_id += 1
            pbar.update(1)

        pbar.close()
        self._vocab_size = next_id

    def encode(self, text: str):
        chunks = self.split_pattern.findall(text)
        out_ids = []
        
        for chunk in chunks:
            # Check the dictionary cache first
            if chunk in self.cache:
                out_ids.extend(self.cache[chunk])
                continue
            
            # Cache miss: break into bytes and process
            chunk_bytes = chunk.encode("utf-8")
            ids = [self.bytes_to_id[bytes([b])] for b in chunk_bytes]
            
            # Merge until no adjacent pair has a valid merge rank
            while len(ids) > 1:
                pairs = [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]
                best_pair = None
                best_rank = float('inf')
                
                # Find the pair with the lowest rank
                for p in pairs:
                    rank = self.merge_ranks.get(p, float('inf'))
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = p
                
                if best_pair is None:
                    break  # No more valid merges for this chunk
                    
                # Apply the best merge to all occurrences in the chunk
                new_id = self.merges[best_pair]
                new_ids = []
                i = 0
                while i < len(ids):
                    if i < len(ids) - 1 and (ids[i], ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(ids[i])
                        i += 1
                ids = new_ids
            
            # Store result in cache and append
            self.cache[chunk] = ids
            out_ids.extend(ids)
            
        return out_ids

    def decode(self, ids):
        out = b"".join(self.id_to_bytes[i] for i in ids)
        return out.decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return self._vocab_size

    @property
    def vocab_size(self):
        return self._vocab_size


def train_tokenizer(train_text: str, vocab_size: int):
    """Train the A1 BPE tokenizer on `train_text` and return it, ready for A3."""
    tok = BPETokenizer()
    tok.train(train_text, vocab_size)
    return tok

## Cell 0c — model classes (Phase 1)

In [13]:
# --- Phase 1 c-1 notebook, cell 21 (verbatim) ---
# ============================================================
# CELL 1 — Decoder-only Transformer (your A3 Q1 code, unchanged except the 2 marked LayerNorm lines)
# Replaces: §9 "Define The Model" (GPT2Config / GPT2LMHeadModel) and the DataCollator cell.
# ============================================================
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------- A3 Q1.1: manual causal multi-head self-attention ----------
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # None means the standard Transformer scaling 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        causal_mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_heads, self.d_head)
        q = q.transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head)
        k = k.transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head)
        v = v.transpose(1, 2)

        scores = q @ k.transpose(-2, -1) * self.attention_scale
        mask = self.causal_mask[:T, :T]
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


# ---------- A3 Q1.2: decoder block and language model ----------
@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 1024
    n_layers: int = 6
    n_heads: int = 8
    n_kv_heads: int = 4          # == n_heads for plain MHA; unused in this version
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"
    attention_scale: float = None
    norm_first: bool = False     # True = pre-norm, False = post-norm


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    pe = torch.zeros(max_len, d_model)
    for a in range(max_len):
        for b in range(d_model):
            i = b // 2
            angle = a / (10000 ** (2 * i / d_model))
            if b % 2 == 0:
                pe[a][b] = math.sin(angle)
            else:
                pe[a][b] = math.cos(angle)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout, attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.norm_first = cfg.norm_first

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))      # FIX: your A3 code used self.ln1 here (ln2 was never used)
        else:
            x = self.ln1(x + self.attn(x))     # FIX: your A3 code used self.ln2 here (ln1 was never used)
            x = self.ln2(x + self.ffn(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) LongTensor of token ids -> logits (B, T, vocab_size)."""
        B, T = idx.shape
        x = self.tok_emb(idx)
        if self.cfg.pos_emb_type == "sinusoidal":
            x = x + self.pos_emb[:T, :]
        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits

## Cell A — load c-1 + tokenizer

In [14]:
# =====================================================================================
# CELL A — load the Phase 1 c-1 decoder and its tokenizer
# Expect <All keys matched successfully>, 110.98M params, vocab 10000, d_model 1024, 8 layers, max_seq_len 2048
# =====================================================================================
import torch, pickle, json, re, os
ROOT = "/kaggle/input/notebooks/rudrajain2006/phase-1-c-1-decoder-only"   # TODO: the folder the find command showed (up to, not including, cs781_phase1_full)
device = torch.device("cuda")

# [c-1] checkpoint folder is legal-gpt2-final (c-6 was model_encdec); class is DecoderLM + LMConfig
tok = pickle.load(open(f"{ROOT}/cs781_phase1_full/legal-gpt2-final/bpe_tokenizer.pkl", "rb"))
ck = torch.load(f"{ROOT}/cs781_phase1_full/legal-gpt2-final/model_final.pt", map_location="cpu", weights_only=False)
cfg = LMConfig(**ck["config"])
lm = DecoderLM(cfg); print(lm.load_state_dict(ck["model"])); lm.to(device).eval()
del ck
print("params", sum(p.numel() for p in lm.parameters())/1e6, "M | vocab", tok.vocab_size,
      "| d_model", cfg.d_model, "| layers", cfg.n_layers, "| d_ff", cfg.d_ff, "| max_seq_len", cfg.max_seq_len,
      "| norm_first", cfg.norm_first, "| dropout", cfg.dropout)

<All keys matched successfully>
params 110.979072 M | vocab 10000 | d_model 1024 | layers 8 | d_ff 4096 | max_seq_len 2048 | norm_first False | dropout 0.1


## Cell B — decoder features check

In [15]:
# =====================================================================================
# CELL B — per-token vectors from the c-1 decoder + sanity check
# [c-1] c-1 has no encoder: the features are the decoder's last hidden states (after ln_f),
#       with the LM head skipped. Each token only sees the tokens to its LEFT (causal).
# =====================================================================================
def features(idx):                        # idx: (B, T) LongTensor, T <= 2048
    x = lm.tok_emb(idx)
    if lm.cfg.pos_emb_type == "sinusoidal":
        x = x + lm.pos_emb[:idx.size(1), :]
    for blk in lm.blocks:
        x = blk(x)
    return lm.ln_f(x)                     # (B, T, 1024)

# Sanity check: the LM head on these features must give the full model's logits
idx = torch.randint(4, cfg.vocab_size, (1, 64), device=device)
with torch.no_grad():
    f = features(idx); diff = (lm.head(f) - lm(idx)).abs().max().item()
print("max diff", diff, tuple(f.shape))   # expect 0.0, (1, 64, 1024)

max diff 0.0 (1, 64, 1024)


## Cell C — CJPE data

In [16]:
# =====================================================================================
# CELL C — Phase 4 (CJPE) data: load train / dev / test cases and check the label field
# One case = one long judgment -> one binary label (0 / 1).
# =====================================================================================
import json
from collections import Counter
from huggingface_hub import hf_hub_download, list_repo_files

REPO = "Exploration-Lab/CS-781-Capstone"
print("Phase 4 files on HF:", [f for f in list_repo_files(REPO, repo_type="dataset") if "Phase_4" in f])

def get4(name):
    """Reads Phase_4_CJPE/<name>. Accepts plain JSON or JSON-lines."""
    p = hf_hub_download(REPO, f"Phase_4_CJPE/{name}", repo_type="dataset")
    with open(p, encoding="utf-8") as f:
        txt = f.read()
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        return [json.loads(line) for line in txt.splitlines() if line.strip()]

_raw = get4("Train/train.json")
_first = _raw[0] if isinstance(_raw, list) else next(iter(_raw.values()))
print("train layout:", type(_raw).__name__, len(_raw), "| first record keys:",
      list(_first) if isinstance(_first, dict) else type(_first).__name__)

TEXT_FIELDS  = ("text", "document", "case_text", "judgment", "sentences")
LABEL_FIELDS = ("label", "labels", "outcome", "decision", "target", "y", "pred")

def as_cases(raw, name):
    """-> {case_id: {"text": str, "label": 0/1 or None}}, in file order.
    Accepts a list of dicts, {case_id: {...}}, a one-key wrapper like {"data": [...]},
    or columns {"id": [...], "text": [...], ("label": [...])}. Other fields (expert_1..5) are ignored.
    Repeated ids (train merges the single- and multi-instance sets):
      same text and label as the first copy -> exact duplicate, dropped;
      different text or label              -> kept as a separate case "<id>#dup<n>" (train/dev),
                                              or dropped with a warning (test: one prediction per id)."""
    while isinstance(raw, dict) and len(raw) == 1 and isinstance(next(iter(raw.values())), (list, dict)):
        raw = next(iter(raw.values()))
    if isinstance(raw, dict) and "id" in raw and isinstance(raw["id"], list):
        keys = list(raw)
        raw = [{k: raw[k][i] for k in keys} for i in range(len(raw["id"]))]
    items = [(d.get("id", n), d) for n, d in enumerate(raw)] if isinstance(raw, list) else list(raw.items())
    cases, n_exact, n_diff, n_conflict = {}, 0, 0, 0
    for k, d in items:
        assert isinstance(d, dict), f"case {k}: unexpected entry {type(d).__name__}"
        k = str(d.get("id", k))
        tf = next((f for f in TEXT_FIELDS if f in d), None)
        assert tf is not None, f"case {k}: no text field (keys {list(d)})"
        text = d[tf]
        if isinstance(text, list):                    # a list of sentences/paragraphs
            text = " ".join(map(str, text))
        lf = next((f for f in LABEL_FIELDS if f in d), None)
        lab = d[lf] if lf is not None else None
        if isinstance(lab, list):
            assert len(lab) == 1, f"case {k}: label is a list {lab}"
            lab = lab[0]
        rec = {"text": str(text), "label": int(lab) if lab is not None else None}
        if k in cases:
            first = cases[k]
            if first["text"] == rec["text"] and first["label"] == rec["label"]:
                n_exact += 1
                continue
            if name == "test":
                print(f"WARNING test id {k} appears twice with different text: keeping the first")
                continue
            n_diff += 1
            n_conflict += first["text"] == rec["text"] and first["label"] != rec["label"]
            j = 1
            while f"{k}#dup{j}" in cases:
                j += 1
            k = f"{k}#dup{j}"
        cases[k] = rec
    print(f"{name}: {len(items)} records -> {len(cases)} cases | exact duplicates dropped {n_exact} | "
          f"repeated ids with different text/label kept {n_diff} (same text, different label: {n_conflict})")
    return cases

train = as_cases(_raw, "train")
dev   = as_cases(get4("Train/dev.json"), "dev")
test  = as_cases(get4("Test/test.json"), "test")
del _raw

for name, split in (("train", train), ("dev", dev), ("test", test)):
    labs = Counter(d["label"] for d in split.values())
    chars = sorted(len(d["text"]) for d in split.values())
    print(f"{name:5s}: {len(split):6d} cases | labels {dict(labs)} | chars median {chars[len(chars)//2]} max {chars[-1]}")
assert all(d["label"] in (0, 1) for s in (train, dev) for d in s.values()), "train/dev labels must be 0/1: check LABEL_FIELDS"
# The test file carries a label field too (placeholder or hidden): never use it, it is cleared here
_tl = Counter(d["label"] for d in test.values())
if any(v is not None for v in _tl):
    print(f"test label field values (ignored, set to None): {dict(_tl)}")
for d in test.values():
    d["label"] = None
print("first test ids:", list(test)[:3])
print("note: write_prediction uses the test ids above, so every test id gets exactly one entry")
print(f"train∩dev ids {len(set(train) & set(dev))} | train∩test {len(set(train) & set(test))}")

Phase 4 files on HF: ['Phase_4_CJPE/Test/test.json', 'Phase_4_CJPE/Train/dev.json', 'Phase_4_CJPE/Train/train.json']
train layout: list 37387 | first record keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
train: 37387 records -> 32305 cases | exact duplicates dropped 5082 | repeated ids with different text/label kept 0 (same text, different label: 0)
dev: 1988 records -> 994 cases | exact duplicates dropped 994 | repeated ids with different text/label kept 0 (same text, different label: 0)
test: 1517 records -> 1517 cases | exact duplicates dropped 0 | repeated ids with different text/label kept 0 (same text, different label: 0)
train:  32305 cases | labels {0: 18920, 1: 13385} | chars median 12120 max 2492701
dev  :    994 cases | labels {0: 497, 1: 497} | chars median 16708 max 260198
test :   1517 cases | labels {1: 762, 0: 755} | chars median 17320 max 233000
test label field values (ignored, set to None): {1: 762, 0: 755}
first test ids: [

## Cell D — tokenise, scorer, writer, baseline

In [17]:
# =====================================================================================
# CELL D — document tokenisation, local scorer, prediction writer, majority baseline
# =====================================================================================
import json, os, zipfile
import numpy as np
from collections import Counter
from tqdm.auto import tqdm

PAD_ID = SPECIAL_TOKENS.index("<pad>")   # 0
UNK_ID = SPECIAL_TOKENS.index("<unk>")   # 1
CLASSES = (0, 1)

def encode_doc(text):
    ids = tok.encode(text)
    return ids if ids else [UNK_ID]

# Tokenise every case once: d["ids"] = token ids of the whole judgment (re-run if you change encode_doc)
for name, split in (("train", train), ("dev", dev), ("test", test)):
    for d in tqdm(split.values(), desc=f"tokenising {name}"):
        d["ids"] = encode_doc(d["text"])

# Length statistics in c-1 tokens, to choose CHUNK_LEN (<= 2048), STRIDE and MAX_CHUNKS in Cell E
for name, split in (("train", train), ("dev", dev), ("test", test)):
    L = np.array([len(d["ids"]) for d in split.values()])
    print(f"{name:5s} tokens: min {L.min()} median {np.median(L):.0f} p90 {np.percentile(L, 90):.0f} "
          f"p99 {np.percentile(L, 99):.0f} max {L.max()} | > 2048: {(L > 2048).mean():.1%}")
for lab in CLASSES:
    L = np.array([len(d["ids"]) for d in train.values() if d["label"] == lab])
    print(f"train label {lab}: {len(L)} cases, median tokens {np.median(L):.0f}")


# -------------------------------------------------------------------------------------
# Local scorer: macro-F1 / macro-P / macro-R over the two classes (= sklearn average="macro")
# -------------------------------------------------------------------------------------
def binary_scores(gold, pred):
    gold, pred = np.asarray(gold), np.asarray(pred)
    per = {}
    for c in CLASSES:
        tp = int(((pred == c) & (gold == c)).sum())
        fp = int(((pred == c) & (gold != c)).sum())
        fn = int(((pred != c) & (gold == c)).sum())
        P = tp / (tp + fp) if tp + fp else 0.0
        R = tp / (tp + fn) if tp + fn else 0.0
        per[c] = (P, R, 2 * P * R / (P + R) if P + R else 0.0, tp + fn)        # last = gold support
    m = lambda i: sum(v[i] for v in per.values()) / len(CLASSES)
    return {"p": m(0), "r": m(1), "f1": m(2), "acc": float((gold == pred).mean()), "per": per}

def evaluate(pred, docs, show=True):
    """pred: {case_id: 0/1}. Scores every case in docs."""
    ids = list(docs)
    res = binary_scores([docs[k]["label"] for k in ids], [pred[k] for k in ids])
    if show:
        print(f"macro-F1 {res['f1']:.4f}  P {res['p']:.4f}  R {res['r']:.4f}  acc {res['acc']:.4f} | " +
              " | ".join(f"class {c}: F1 {v[2]:.3f} (n={v[3]})" for c, v in res["per"].items()))
    return res

def confusion(pred, docs):
    """Rows = gold, columns = predicted."""
    M = np.zeros((2, 2), dtype=int)
    for k, d in docs.items():
        M[d["label"], pred[k]] += 1
    print(f"confusion (rows gold, cols pred)\n        pred 0  pred 1\ngold 0 {M[0,0]:7d} {M[0,1]:7d}\ngold 1 {M[1,0]:7d} {M[1,1]:7d}")
    return M

try:   # cross-check against sklearn once
    from sklearn.metrics import f1_score
    _g, _p = [0, 1, 1, 0, 1], [0, 1, 0, 0, 0]
    assert abs(binary_scores(_g, _p)["f1"] - f1_score(_g, _p, average="macro")) < 1e-9
    print("scorer matches sklearn macro-F1")
except ImportError:
    pass


# -------------------------------------------------------------------------------------
# Writer: prediction.json = LIST of {"id", "pred"} with integer pred, in test order, zipped
# -------------------------------------------------------------------------------------
def write_prediction(pred, test_docs, out_dir="/kaggle/working"):
    os.makedirs(out_dir, exist_ok=True)
    out = []
    for case_id in test_docs:
        p = int(pred[case_id])                       # numpy ints / bools -> plain int
        assert p in (0, 1), f"{case_id}: pred {p!r} is not 0/1"
        out.append({"id": case_id, "pred": p})
    assert len(out) == len(test_docs) and len({o["id"] for o in out}) == len(out), "one entry per test case"
    assert set(pred) >= set(test_docs), "missing test ids"
    path = os.path.join(out_dir, "prediction.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False)
    back = json.load(open(path))
    assert isinstance(back, list) and all(type(o["pred"]) is int and set(o) == {"id", "pred"} for o in back)
    zpath = os.path.join(out_dir, "submission.zip")
    with zipfile.ZipFile(zpath, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    with zipfile.ZipFile(zpath) as z:
        assert z.namelist() == ["prediction.json"], z.namelist()
    print(f"{len(out)} cases | predicted label counts {dict(Counter(o['pred'] for o in out))} -> {zpath}")


# -------------------------------------------------------------------------------------
# Majority baseline: checks the scorer and writer end to end before any modelling
# -------------------------------------------------------------------------------------
majority = Counter(d["label"] for d in train.values()).most_common(1)[0][0]
print("majority label:", majority)
evaluate({k: majority for k in dev}, dev)
write_prediction({k: majority for k in test}, test, out_dir="/kaggle/working/baseline")   # format check only; don't spend a slot on it

tokenising train:   0%|          | 0/32305 [00:00<?, ?it/s]

tokenising dev:   0%|          | 0/994 [00:00<?, ?it/s]

tokenising test:   0%|          | 0/1517 [00:00<?, ?it/s]

train tokens: min 55 median 2712 p90 8660 p99 25828 max 545646 | > 2048: 60.4%
dev   tokens: min 220 median 4262 p90 10448 p99 30676 max 71037 | > 2048: 87.0%
test  tokens: min 107 median 4442 p90 10997 p99 27443 max 64625 | > 2048: 87.1%
train label 0: 18920 cases, median tokens 2688
train label 1: 13385 cases, median tokens 2743
scorer matches sklearn macro-F1
majority label: 0
macro-F1 0.3333  P 0.2500  R 0.5000  acc 0.5000 | class 0: F1 0.667 (n=497) | class 1: F1 0.000 (n=497)
1517 cases | predicted label counts {0: 1517} -> /kaggle/working/baseline/submission.zip


## Cell E0a — shared plumbing: decoder path, chunking, DocModel

In [18]:
# =====================================================================================
# CELL E0a — shared plumbing for the c-1 Phase 4 (CJPE) experiments (part 1 of 2)
# Run E0a then E0b once after Cells 0, A, B, C, D. Then run the technique cell (E BitFit).
#
# Long documents: a judgment is cut into chunks of CHUNK_LEN tokens every STRIDE tokens,
# DOC_REPR picks which chunks represent the case (whole document, beginning, end, ...),
# each chunk goes through the c-1 decoder, the pool turns each chunk into one vector, and
# the head turns the case's sequence of chunk vectors into 2 logits.
# Nothing here is a model design or a hyperparameter.
# Lines marked [c-1] are where this differs from the c-6 Phase 4 version.
# =====================================================================================
import math, time, random
from collections import defaultdict, Counter
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm

N_CLASSES = 2
FEAT_DIM = cfg.d_model   # [c-1] 1024 (c-6 was 560)
# [c-1] nothing to drop: c-1 is decoder-only. lm.head (tied to tok_emb) is simply never called.

GRAD_CHECKPOINT = False   # memory switch (not a hyperparameter: same maths, slower). Set in Cell E.


# -------------------------------------------------------------------------------------
# [c-1] decoder path (same as Cell B features(), on any backbone object)
#   c-1 is causal, so a token only sees tokens to its LEFT. With RIGHT padding the real
#   tokens never see the pads, so no padding mask is needed (checked at the end of E0b).
#   key_pad is accepted only to keep the same call shape as c-6, and is not used.
# -------------------------------------------------------------------------------------
def decoder_forward(backbone, idx, key_pad=None):
    T = idx.size(1)
    assert T <= backbone.cfg.max_seq_len, f"chunk longer than {backbone.cfg.max_seq_len}"
    x = backbone.tok_emb(idx)
    if backbone.cfg.pos_emb_type == "sinusoidal":
        x = x + backbone.pos_emb[:T, :]
    for blk in backbone.blocks:                  # 8 causal decoder Blocks
        if GRAD_CHECKPOINT and torch.is_grad_enabled() and backbone.training:
            x = checkpoint(blk, x, use_reentrant=False)
        else:
            x = blk(x)
    return backbone.ln_f(x)                      # (B, T, 1024); LM head skipped


# -------------------------------------------------------------------------------------
# Chunking and document representation
#   make_windows: full-length windows of `chunk_len` tokens every `stride` tokens; the last
#     window ends at the document end. A short document is one shorter chunk.
#   DOC_REPR (which chunks stand for the case; max_chunks caps how many):
#     "all"        every chunk (max_chunks ignored; long cases cost the most memory)
#     "first"      the first max_chunks chunks (= plain truncation when max_chunks is 1)
#     "last"       the last max_chunks chunks (the end of a judgment, where the reasoning is)
#     "first_last" half from the start, the rest from the end
#     "spread"     max_chunks chunks evenly spaced over the whole document
#   Chunks are always kept in document order.
# -------------------------------------------------------------------------------------
DOC_REPRS = ("all", "first", "last", "first_last", "spread")

def make_windows(n, window, stride):
    if n <= window:
        return [(0, n)]
    starts = list(range(0, n - window, stride)) + [n - window]
    return [(s, s + window) for s in starts]


def select_chunks(wins, doc_repr, max_chunks):
    n = len(wins)
    if doc_repr == "all" or n <= max_chunks:
        return wins
    if doc_repr == "first":
        return wins[:max_chunks]
    if doc_repr == "last":
        return wins[-max_chunks:]
    if doc_repr == "first_last":
        h = (max_chunks + 1) // 2
        return wins[:h] + wins[n - (max_chunks - h):] if max_chunks > h else wins[:h]
    if doc_repr == "spread":
        idx = sorted({round(i * (n - 1) / max(1, max_chunks - 1)) for i in range(max_chunks)})
        return [wins[i] for i in idx]
    raise ValueError(doc_repr)


def build_doc_examples(docs, *, chunk_len, stride, doc_repr, max_chunks):
    """docs: {case_id: doc} from Cells C/D. -> list of {"id", "chunks": [[ids], ...], "label"}.
    label is None for test cases."""
    assert chunk_len <= cfg.max_seq_len, f"chunk_len > max_seq_len {cfg.max_seq_len}"   # [c-1] 2048
    assert doc_repr in DOC_REPRS, doc_repr
    examples = []
    for case_id, d in tqdm(docs.items(), desc=f"chunking ({doc_repr})", leave=False):
        ids = d["ids"]
        wins = select_chunks(make_windows(len(ids), chunk_len, stride), doc_repr, max_chunks)
        examples.append({"id": case_id, "chunks": [ids[s:e] for s, e in wins], "label": d["label"]})
    n = [len(ex["chunks"]) for ex in examples]
    print(f"{len(examples)} cases | chunks per case: mean {sum(n)/len(n):.2f} max {max(n)}")
    return examples


# -------------------------------------------------------------------------------------
# Model wrapper: chunks -> decoder -> pool per chunk -> (B, C, pooled) -> head -> (B, 2)
#   pool: function/module, (L, 1024) real tokens of ONE chunk -> one vector
#   head: module, forward(Z, lengths): Z (B, C, pooled_dim) chunk vectors per case in
#         document order (rows past lengths[b] are zero padding), lengths (B,) CPU LongTensor
#         -> logits (B, 2)
# -------------------------------------------------------------------------------------
class DocModel(nn.Module):
    def __init__(self, backbone, pool, head):
        super().__init__()
        self.backbone, self.pool, self.head = backbone, pool, head
    def forward(self, ids, key_pad, n_chunks):
        """ids, key_pad: (sum C, T) for every chunk of every case in the batch; n_chunks: list, chunks per case."""
        h = decoder_forward(self.backbone, ids, key_pad)                 # [c-1] (sum C, T, 1024)
        lens = (~key_pad).sum(1).tolist()
        vecs = torch.stack([self.pool(h[i, :L]) for i, L in enumerate(lens)])      # right padding: real tokens are h[i, :L]
        per_case = list(vecs.split(n_chunks))
        Z = nn.utils.rnn.pad_sequence(per_case, batch_first=True)       # (B, C, pooled)
        return self.head(Z, torch.tensor(n_chunks))                     # (B, 2)

## Cell E0b — shared plumbing: batching, training (1 h cap), prediction, submission

In [19]:
# =====================================================================================
# CELL E0b — shared plumbing for the c-1 Phase 4 (CJPE) experiments (part 2 of 2)
# Batching, loss, prediction, training loop, error cases, submission. Run right after E0a.
# =====================================================================================

def identity_probe():
    """First 64 tokens of a training case, for the padding check below."""
    d = next(iter(train.values()))
    return torch.tensor([d["ids"][:64]], device=device)


def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    print(f"trainable {tr/1e6:.3f}M / total {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    return tr, tot


def make_doc_batches(examples, batch_size, shuffle):
    """Batches of `batch_size` CASES, grouped by similar total tokens (sorted inside pools of 50 batches)."""
    exs = list(examples)
    if shuffle:
        random.shuffle(exs)
    out, pool = [], batch_size * 50
    size = lambda ex: (len(ex["chunks"]), max(map(len, ex["chunks"])))
    for i in range(0, len(exs), pool):
        chunk = sorted(exs[i:i + pool], key=size)
        out += [chunk[j:j + batch_size] for j in range(0, len(chunk), batch_size)]
    if shuffle:
        random.shuffle(out)
    return out


def to_doc_tensors(batch):
    """All chunks of all cases in the batch, padded to the longest chunk, plus a padding mask."""
    chunks = [c for ex in batch for c in ex["chunks"]]
    T = max(map(len, chunks))
    ids = torch.full((len(chunks), T), PAD_ID, dtype=torch.long)
    key_pad = torch.ones((len(chunks), T), dtype=torch.bool)             # True = padding
    for i, c in enumerate(chunks):
        ids[i, :len(c)] = torch.tensor(c)
        key_pad[i, :len(c)] = False
    b = {"ids": ids.to(device), "key_pad": key_pad.to(device),
         "n_chunks": [len(ex["chunks"]) for ex in batch]}
    if batch[0]["label"] is not None:
        b["labels"] = torch.tensor([ex["label"] for ex in batch], device=device)
    if "soft" in batch[0]:                                               # teacher logits (KD)
        b["soft"] = torch.stack([ex["soft"] for ex in batch]).to(device)
    return b


def ce_loss(logits, labels, class_weights=None):
    """Plain case-level cross-entropy on gold labels."""
    return F.cross_entropy(logits.float(), labels, weight=class_weights)


def make_class_weights(scheme):
    """'inverse' (1/count), 'sqrt_inverse' (1/sqrt(count)) or 'none', from the train label counts,
    normalised to mean 1 so the loss scale stays similar."""
    assert scheme in ("inverse", "sqrt_inverse", "none"), "set WEIGHT_SCHEME"
    cnt = Counter(d["label"] for d in train.values())
    counts = torch.tensor([cnt.get(c, 0) for c in CLASSES], dtype=torch.float)
    if scheme == "none":
        w = None
    else:
        w = 1.0 / counts if scheme == "inverse" else 1.0 / counts.sqrt()
        w = w / w.mean()
    for c, n in zip(CLASSES, counts.tolist()):
        print(f"  class {c}: train n={int(n)}  weight={'1.000' if w is None else f'{w[c].item():.3f}'}")
    return None if w is None else w.to(device)


# -------------------------------------------------------------------------------------
# Prediction. predict_probs -> {case_id: P(label 1)}; predict -> {case_id: 0/1}.
# threshold: decision rule on P(label 1) (0.5 = argmax). Calibration/post-processing is yours.
# -------------------------------------------------------------------------------------
@torch.no_grad()
def predict_probs(model, examples, batch_size):
    model.eval()
    out = {}
    for batch in tqdm(make_doc_batches(examples, batch_size, shuffle=False), desc="predict", leave=False):
        b = to_doc_tensors(batch)
        with torch.autocast("cuda", dtype=torch.float16):
            p1 = model(b["ids"], b["key_pad"], b["n_chunks"]).float().softmax(-1)[:, 1].tolist()
        for ex, p in zip(batch, p1):
            out[ex["id"]] = p
    assert len(out) == len(examples), "every case needs a prediction"
    return out


def predict(model, examples, batch_size, threshold=0.5):
    return {k: int(p >= threshold) for k, p in predict_probs(model, examples, batch_size).items()}


def warmup_cosine(total_steps, warmup_frac):
    """LR multiplier per optimiser step: linear warm-up over warmup_frac of the steps, then cosine to 0."""
    warm = max(1, int(warmup_frac * total_steps))
    def f(step):
        if step < warm:
            return (step + 1) / warm
        return 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total_steps - warm)))
    return f


# -------------------------------------------------------------------------------------
# Training loop. loss_fn(logits, batch_tensors) comes from the technique cell.
# param_groups: [{"params": [...], "lr": ...}, ...] so adapters and head can use different LRs.
# grad_accum: optimiser step every grad_accum batches (bigger effective batch, same memory).
# Keeps the epoch with the best dev macro-F1 (at `threshold`).
# -------------------------------------------------------------------------------------
def train_model(model, train_ex, dev_ex, dev_docs, loss_fn, *, lr, epochs, batch_size,
                weight_decay=0.0, grad_clip=1.0, lr_lambda=None, save_path="best_trainable.pt",
                param_groups=None, grad_accum=1, threshold=0.5, max_train_minutes=None):
    """max_train_minutes: hard cap on time spent in training batches (dev scoring not counted).
    When it is reached, the epoch stops after the batch in progress, the pending gradients
    are applied, dev is scored, the weights are saved, and training ends."""
    params = [p for p in model.parameters() if p.requires_grad]
    if param_groups is None:
        param_groups = [{"params": params, "lr": lr}]
    grouped = {id(p) for g in param_groups for p in g["params"]}
    assert grouped == {id(p) for p in params}, "param_groups must cover every trainable parameter exactly"
    opt = torch.optim.AdamW(param_groups, lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda) if lr_lambda else None
    scaler = torch.amp.GradScaler("cuda")
    torch.cuda.reset_peak_memory_stats()
    t0, best_f1, best_res, log, skipped = time.time(), -1.0, None, [], 0
    cap_s = max_train_minutes * 60 if max_train_minutes is not None else None
    used_s, capped = 0.0, False
    def opt_step():
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, grad_clip)
        scaler.step(opt); scaler.update()
        opt.zero_grad(set_to_none=True)
        if sched: sched.step()
    for epoch in range(epochs):
        model.train(); tot, nb, seen, pending = 0.0, 0, 0, False
        batches = make_doc_batches(train_ex, batch_size, shuffle=True)
        pbar = tqdm(batches, desc=f"epoch {epoch + 1}/{epochs}")
        opt.zero_grad(set_to_none=True)
        t_ep = time.time()
        for i, batch in enumerate(pbar):
            if cap_s is not None and used_s + (time.time() - t_ep) >= cap_s:
                capped = True
                break
            seen += 1
            b = to_doc_tensors(batch)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(b["ids"], b["key_pad"], b["n_chunks"])
            loss = loss_fn(logits.float(), b)
            if not torch.isfinite(loss):
                skipped += 1
                continue
            scaler.scale(loss / grad_accum).backward(); pending = True
            if (i + 1) % grad_accum == 0 or i + 1 == len(batches):
                opt_step(); pending = False
            tot += loss.item(); nb += 1
            pbar.set_postfix(loss=f"{tot / nb:.4f}", lr=f"{opt.param_groups[0]['lr']:.2e}", skipped=skipped,
                             min=f"{(used_s + time.time() - t_ep) / 60:.1f}")
        if pending:                                  # cap hit mid-accumulation: apply what was accumulated
            opt_step()
        used_s += time.time() - t_ep
        pbar.close()
        if capped:
            print(f"time cap {max_train_minutes} min reached in epoch {epoch + 1}: trained on {seen}/{len(batches)} "
                  f"batches (~{min(seen * batch_size, len(train_ex))}/{len(train_ex)} cases)")
        res = evaluate(predict(model, dev_ex, batch_size, threshold), dev_docs, show=False)
        log.append({"epoch": epoch + 1, "train_loss": tot / max(1, nb), "dev_f1": res["f1"],
                    "dev_p": res["p"], "dev_r": res["r"], "dev_acc": res["acc"], "skipped": skipped,
                    "batches_seen": seen, "batches_total": len(batches), "train_min_so_far": round(used_s / 60, 1)})
        print(log[-1])
        if res["f1"] > best_f1:
            best_f1, best_res = res["f1"], res
            torch.save({n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}, save_path)
        if capped:
            break
    print(f"best dev macro-F1 {best_f1:.4f} (P {best_res['p']:.4f}, R {best_res['r']:.4f}) | "
          f"train time {used_s / 60:.1f} min (+ dev scoring, total {(time.time() - t0) / 60:.1f} min) | peak GPU mem {torch.cuda.max_memory_allocated() / 2**30:.2f} GB")
    return log, best_res


@torch.no_grad()
def time_inference(model, examples, batch_size):
    """Wall-clock seconds to predict a set (for the efficiency table in the report)."""
    torch.cuda.synchronize(); t = time.time()
    predict_probs(model, examples, batch_size)
    torch.cuda.synchronize(); return time.time() - t


def error_cases(pred, probs, docs, k=5, chars=300):
    """A few false positives (gold 0, pred 1) and false negatives (gold 1, pred 0), most confident first."""
    for name, g, p in (("FALSE POSITIVES", 0, 1), ("FALSE NEGATIVES", 1, 0)):
        ids = [i for i, d in docs.items() if d["label"] == g and pred[i] == p]
        ids.sort(key=lambda i: -abs(probs[i] - 0.5))
        print(f"\n{name}: {len(ids)}")
        for i in ids[:k]:
            t = docs[i]["text"]
            print(f"- {i} | P(1)={probs[i]:.3f} | {len(docs[i]['ids'])} tokens\n  start: {t[:chars]!r}\n  end:   {t[-chars:]!r}")


def write_submission(model, build_kwargs, batch_size, best_path=None, out_dir="/kaggle/working", threshold=0.5):
    """Reload the best trainable weights, predict test, write prediction.json + submission.zip."""
    if best_path:
        model.load_state_dict(torch.load(best_path), strict=False)
    test_ex = build_doc_examples(test, **build_kwargs)
    write_prediction(predict(model, test_ex, batch_size, threshold), test, out_dir=out_dir)


# [c-1] Check: decoder_forward equals Cell B features(), and RIGHT padding does not change real tokens (causal)
lm.eval()
with torch.no_grad():
    _p = identity_probe()
    _ref = features(_p)
    _d1 = (decoder_forward(lm, _p) - _ref).abs().max().item()
    _pad = torch.cat([_p, torch.full((1, 7), PAD_ID, device=device)], 1)
    _d2 = (decoder_forward(lm, _pad)[:, :_p.size(1)] - _ref).abs().max().item()
print(f"decoder check: no-pad diff {_d1:.2e}, right-padded diff {_d2:.2e} (both should be ~0)")
assert _d1 < 1e-4 and _d2 < 1e-4, "right padding changes the real tokens' features"

decoder check: no-pad diff 0.00e+00, right-padded diff 9.54e-07 (both should be ~0)


## Cell E — BitFit

In [20]:
# =====================================================================================
# CELL E — BitFit on the c-1 decoder for Court Judgment Prediction (CJPE)   — ref [8] Ben Zaken et al. 2022
# Run after Cells 0, A, B, C, D, E0a and E0b. To run it again from scratch, re-run Cell A,
# E0a and E0b first (fresh `lm`, nothing left trainable from an earlier run).
#
# BitFit = freeze every weight matrix and train only the bias terms, plus the head.
# What the c-1 decoder has (from your Phase 1 code):
#   - attention q_proj / k_proj / v_proj / out_proj: bias=False -> NO attention biases exist
#   - ffn.net[0] (1024 -> d_ff) and ffn.net[2] (d_ff -> 1024) biases, per block
#   - LayerNorm ln1 / ln2 biases per block, and ln_f
#   - tok_emb and the tied LM head have no bias
#   With d_ff = 4096 that is 8 x (4096 + 1024 + 1024 + 1024) + 1024 = 58,368 bias values
#   (vs ~111M parameters). The listing printed below shows the exact names and counts.
#
# Filled in from your earlier cells: [S] all bias terms (the BitFit paper's setting),
# [P] mean pooling over a chunk's tokens, [N] mean over the case's chunks, then your
# Dropout + Linear head. Every value marked None is yours to set; the asserts stop you.
# =====================================================================================

# -------------------------------------------------------------------------------------
# [H] Settings and hyperparameters (None = not set)
# -------------------------------------------------------------------------------------
CHUNK_LEN        = 1024
STRIDE           = 512
DOC_REPR         = "spread"
MAX_CHUNKS       = 8

LR               = 1e-4
HEAD_LR          = 3e-4
EPOCHS           = 1
MAX_TRAIN_MINUTES = 60
BATCH_SIZE       = 4
GRAD_ACCUM       = 4
WEIGHT_DECAY     = 0.0
WARMUP_FRAC      = 0.05

WEIGHT_SCHEME    = "sqrt_inverse"
DROPOUT          = 0.1
THRESHOLD        = 0.5

SEED             = 42
GRAD_CHECKPOINT  = True
for _n in ["CHUNK_LEN", "STRIDE", "DOC_REPR", "MAX_CHUNKS", "LR", "EPOCHS", "BATCH_SIZE",
           "GRAD_ACCUM", "WEIGHT_DECAY", "WEIGHT_SCHEME", "DROPOUT", "THRESHOLD"]:
    assert globals()[_n] is not None, f"set {_n}"
assert 0 < STRIDE <= CHUNK_LEN <= cfg.max_seq_len, "need 0 < STRIDE <= CHUNK_LEN <= 2048"
assert DOC_REPR in DOC_REPRS, DOC_REPR
BUILD = dict(chunk_len=CHUNK_LEN, stride=STRIDE, doc_repr=DOC_REPR, max_chunks=MAX_CHUNKS)
random.seed(SEED); torch.manual_seed(SEED)
RUN_NAME = f"c1_bitfit_{DOC_REPR}"
SAVE = f"/kaggle/working/{RUN_NAME}_best_trainable.pt"

CLASS_WEIGHTS = make_class_weights(WEIGHT_SCHEME)


# -------------------------------------------------------------------------------------
# Everything frozen; list the decoder's bias terms
# -------------------------------------------------------------------------------------
for p in lm.parameters():
    p.requires_grad = False

dec_bias = {n: p for n, p in lm.named_parameters() if n.endswith(".bias")}
groups = defaultdict(int)
for n, p in dec_bias.items():
    key = ("ffn_in" if ".ffn.net.0." in n else "ffn_out" if ".ffn.net.2." in n else n.split(".")[-2])   # ffn_in / ffn_out / ln1 / ln2 / ln_f
    groups[key] += p.numel()
print(f"{len(dec_bias)} decoder bias tensors, {sum(groups.values()):,} values:", dict(groups))
for n in list(dec_bias)[:5]:
    print("  e.g.", n, tuple(dec_bias[n].shape))


# -------------------------------------------------------------------------------------
# [S] BitFit selection: every bias term (FFN + LayerNorm), as in the BitFit paper.
#   To try a subset instead (only FFN, only LayerNorm, ...), filter `dec_bias` by name here
#   and record it in the report.
# -------------------------------------------------------------------------------------
for n, p in dec_bias.items():
    p.requires_grad = True

n_bias_tr = sum(p.numel() for p in dec_bias.values() if p.requires_grad)
n_other_tr = sum(p.numel() for n, p in lm.named_parameters() if p.requires_grad and n not in dec_bias)
print(f"trainable decoder biases: {n_bias_tr:,} | other trainable decoder params: {n_other_tr:,}")
assert n_bias_tr > 0, "no bias is trainable"
assert n_other_tr == 0, "BitFit trains biases only; a non-bias decoder parameter is trainable"


# -------------------------------------------------------------------------------------
# [P] Chunk representation: h is (L, 1024), the real tokens of ONE chunk -> mean (your pooling)
# -------------------------------------------------------------------------------------
def pool_chunk(h):
    return h.mean(0)


# -------------------------------------------------------------------------------------
# [N] Head: average the case's real chunk vectors, then your Dropout + Linear (1024 -> 2)
#   Z: (B, C, 1024), rows past lengths[b] are zero padding; lengths: (B,) on CPU
# -------------------------------------------------------------------------------------
class DocHead(nn.Module):
    def __init__(self, in_dim, n_classes, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_dim, n_classes)
        )
    def forward(self, Z, lengths):
        mask = (torch.arange(Z.size(1))[None, :] < lengths[:, None]).to(Z.device, Z.dtype)   # (B, C)
        doc = (Z * mask[..., None]).sum(1) / lengths.to(Z.device, Z.dtype)[:, None]          # mean of real chunks
        return self.net(doc)                                                                 # (B, 2)

head = DocHead(FEAT_DIM, N_CLASSES, DROPOUT)
model = DocModel(lm, pool_chunk, head).to(device)
tr_params, tot_params = count_params(model)

head_params = list(model.head.parameters())
bias_params = [p for p in dec_bias.values() if p.requires_grad]
param_groups = [{"params": bias_params, "lr": LR},
                {"params": head_params, "lr": LR if HEAD_LR is None else HEAD_LR}]


# -------------------------------------------------------------------------------------
# Data, training, scoring (plumbing from E0a/E0b). Model selection on dev macro-F1.
# -------------------------------------------------------------------------------------
train_ex = build_doc_examples(train, **BUILD)
dev_ex = build_doc_examples(dev, **BUILD)
print(f"train cases {len(train_ex)} | dev cases {len(dev_ex)}")

_steps = EPOCHS * math.ceil(math.ceil(len(train_ex) / BATCH_SIZE) / GRAD_ACCUM)   # full epoch; if the time cap stops it early the cosine does not reach 0
LR_LAMBDA = warmup_cosine(_steps, WARMUP_FRAC) if WARMUP_FRAC is not None else None

log, best_res = train_model(
    model, train_ex, dev_ex, dev,
    loss_fn=lambda logits, b: ce_loss(logits, b["labels"], CLASS_WEIGHTS),
    lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY,
    lr_lambda=LR_LAMBDA, save_path=SAVE, param_groups=param_groups,
    grad_accum=GRAD_ACCUM, threshold=THRESHOLD, max_train_minutes=MAX_TRAIN_MINUTES)


# -------------------------------------------------------------------------------------
# Evidence for the report: best-epoch dev scores, confusion matrix, error cases, cost
# -------------------------------------------------------------------------------------
model.load_state_dict(torch.load(SAVE), strict=False)
dev_probs = predict_probs(model, dev_ex, BATCH_SIZE)
dev_pred = {k: int(p >= THRESHOLD) for k, p in dev_probs.items()}
dev_res = evaluate(dev_pred, dev)
confusion(dev_pred, dev)
error_cases(dev_pred, dev_probs, dev)
inf_s = time_inference(model, dev_ex, BATCH_SIZE)
print(f"\n{RUN_NAME} | trained on {log[-1]['batches_seen']}/{log[-1]['batches_total']} batches in {log[-1]['train_min_so_far']} min | trainable {tr_params:,} / {tot_params:,} | chunk {CHUNK_LEN}/{STRIDE} {DOC_REPR} x{MAX_CHUNKS} | "
      f"dev macro-F1 {dev_res['f1']:.4f} | dev inference {inf_s:.1f} s")

# Predict test with the best weights and package /kaggle/working/submission.zip
write_submission(model, BUILD, BATCH_SIZE, best_path=SAVE, threshold=THRESHOLD)

  class 0: train n=18920  weight=0.914
  class 1: train n=13385  weight=1.086
33 decoder bias tensors, 58,368 values: {'ln1': 8192, 'ln2': 8192, 'ffn_in': 32768, 'ffn_out': 8192, 'ln_f': 1024}
  e.g. blocks.0.ln1.bias (1024,)
  e.g. blocks.0.ln2.bias (1024,)
  e.g. blocks.0.ffn.net.0.bias (4096,)
  e.g. blocks.0.ffn.net.2.bias (1024,)
  e.g. blocks.1.ln1.bias (1024,)
trainable decoder biases: 58,368 | other trainable decoder params: 0
trainable 0.060M / total 110.98M (0.05%)


chunking (spread):   0%|          | 0/32305 [00:00<?, ?it/s]

32305 cases | chunks per case: mean 4.77 max 8


chunking (spread):   0%|          | 0/994 [00:00<?, ?it/s]

994 cases | chunks per case: mean 6.36 max 8
train cases 32305 | dev cases 994


epoch 1/1:   0%|          | 0/8077 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 704.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 650.81 MiB is free. Including non-PyTorch memory, this process has 13.92 GiB memory in use. Of the allocated memory 13.28 GiB is allocated by PyTorch, and 520.08 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)